In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sbs
import math as math
import os
from os import listdir
pd.options.mode.use_inf_as_na = True
import tkinter
from tkinter import filedialog
import scipy as scp

In [ ]:
tkinter.Tk().withdraw()
filepath = filedialog.askdirectory()

In [ ]:
# df_files = {}
# i = 0

# for root, dirs, files in os.walk(filepath):
#     if ".csv" in str(files) and "Trial" in str(files):
#         for f in listdir(path=root):
#             if ".csv" in str(f):
#                     a = pd.read_csv(root + '\\' + f)
#                     y = a.xs('Y (cm)', axis=1)
#                     x = a.xs('X (cm)', axis=1)
#                     speed = a.xs('SPEED#wcentroid (cm/s)', axis=1)
#                     vy = a.xs('VY (cm/s)', axis=1)
#                     frame = a.frame
#                     df_files[str(i)] = pd.DataFrame({"Y" : y, "X" : x, "Speed" : speed, "VY" : vy, "Frame" : frame, "Trial" : str(f), "Condition" : str(root.rsplit('\\')[-2])})
#                     i = i+1

# complete_df = pd.concat(df_files)

In [ ]:
df_files = {}
i = 0

for root, dirs, files in os.walk(filepath):
    if ".csv" in str(files) and "Trial" in str(files):
        for f in listdir(path=root):
            if ".csv" in str(f):
                    a = pd.read_csv(root + '\\' + f)
                    y = a.xs('Y (cm)', axis=1)
                    x = a.xs('X (cm)', axis=1)
                    speed = a.xs('SPEED#wcentroid (cm/s)', axis=1)
                    vy = a.xs('VY (cm/s)', axis=1)
                    frame = a.frame
                    df_files[str(i)] = pd.DataFrame({"Y" : y, "X" : x, "Speed" : speed, "VY" : vy, "Frame" : frame, "Trial" : str(root.rsplit('\\')[-1]), "Condition" : str(root.rsplit('\\')[-2])})
                    i = i+1

complete_df = pd.concat(df_files)

In [ ]:
#Cleaning the data: ridding it of infs
clean_complete_df = complete_df.dropna()

clean_complete_df.head()


In [ ]:
clean_complete_df = clean_complete_df[clean_complete_df["Speed"] < 2.0]

In [ ]:
clean_complete_df = clean_complete_df[clean_complete_df["Speed"] > 0.5]

In [ ]:
conditions = clean_complete_df['Condition'].unique().tolist()

trials_dict = {}

for cond in conditions:
    key = f'{cond}'
    sub_df = clean_complete_df[clean_complete_df['Condition'] == cond] 
    value = sub_df['Trial'].unique().tolist()
    trials_dict[key] = value    



In [ ]:
#To plot speed between the conditions

speed_grouped = clean_complete_df.groupby(['Condition', 'Trial']).Speed.mean()

In [ ]:
#speed_grouped = speed_grouped.reset_index()
#speed_grouped

In [ ]:
sbs.boxplot(data = clean_complete_df, x = "Condition", y = "Speed")

In [ ]:
df_300 = clean_complete_df[clean_complete_df["Frame"]<300]

In [ ]:
speed_grouped_300 = df_300.groupby(["Condition", "Trial"]).Speed.mean()

In [ ]:
speed_grouped_300 = speed_grouped_300.reset_index()

In [ ]:
ax = sbs.boxplot(data = df_300, x = "Condition", y = "Speed")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')  # Rotate labels by 45 degrees

plt.show()

In [ ]:
for cond in conditions:
    df_inst = df_300[df_300["Condition"] == cond]
        # Define the bin size
        # Define the ranges for x and y axes
    x_min, x_max = 0, 30
    y_min, y_max = 0, 30

    # Define the bin size
    bin_size = 0.5

    # Calculate the number of bins in x and y directions
    x_bins = np.arange(x_min, x_max + bin_size, bin_size)
    y_bins = np.arange(y_min, y_max + bin_size, bin_size)

    # Bin the data
    df_inst['x_bin'] = pd.cut(df_inst['X'], bins=x_bins, labels=False)
    df_inst['y_bin'] = pd.cut(df_inst['Y'], bins=y_bins, labels=False)

    # Count the number of points in each bin
    heatmap_data = df_inst.groupby(['x_bin', 'y_bin']).size().unstack(fill_value=0)

    # Convert to numpy array for heatmap plotting
    heatmap_array = heatmap_data.values

    # Normalise to total number of points
    heatmap_array = heatmap_array/heatmap_array.sum()


    # Plot the heatmap
    plt.figure(figsize=(10, 8))
    plt.imshow(heatmap_array, cmap='hot', interpolation='nearest', origin='lower')
    plt.colorbar(label='Number of Points - Normalized')
    plt.title(cond)
    plt.xlabel('X Bin')
    plt.ylabel('Y Bin')
    plt.show()


In [ ]:
trials = df_300['Trial'].unique()

In [ ]:
for cond in conditions:
    df_inst = df_300[df_300["Condition"] == cond]
    for trial in trials:
        df_trial_inst = df_inst[df_inst['Trial'] == trial]
        sbs.scatterplot(data = df_trial_inst, x = "X", y = "Y", hue = "Frame")
        plt.title(cond + ' ' + trial)
        plt.xlim(0,30)
        plt.ylim(0,30)
        plt.show()

In [ ]:
#Inserting a distance column
clean_complete_df.insert(4, 'Distance', 0)

In [ ]:
for cond in conditions:
    for trial in trials_dict[f'{cond}']:
        sub_df = clean_complete_df[(clean_complete_df['Condition'] == cond) & (clean_complete_df['Trial'] == trial)]
        #Calculating centroid position

        x_centroid_list = []
        y_centroid_list = []

        frame_list = sub_df['Frame'].values.tolist()
        for f in range(0, int(max(frame_list))):
            cropped_pos = sub_df.loc[sub_df['Frame'] == f]
            av_x = cropped_pos['X'].mean()
            x_centroid_list.append(av_x)
            av_y = cropped_pos['Y'].mean()
            y_centroid_list.append(av_y)

        coordlist = list(zip(x_centroid_list, y_centroid_list))
        t = np.arange(np.size(x_centroid_list))

        x_1 = 14
        y_1 = 14

        for f in sub_df.index:
            x_2 = sub_df.X[f]
            y_2 = sub_df.Y[f]
            dist = math.sqrt((x_2-x_1)**2 + (y_2-y_1)**2)
            clean_complete_df.Distance[f] = dist



In [ ]:
distance_grouped = clean_complete_df.groupby(['Condition', 'Trial']).Distance.mean()
distance_grouped = distance_grouped.reset_index()

In [ ]:
#distance_grouped = distance_grouped.pivot(index='Trial', columns='Condition', values = 'Speed')

In [ ]:
distance_grouped

In [ ]:
#for index in distance_grouped.index:
    #sbs.boxenplot(data = distance_grouped, x = distance_grouped[index], y = 'Condition')

ax = sbs.boxplot(data=distance_grouped, x = "Condition", y = "Distance")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')  # Rotate labels by 45 degrees

plt.show()

#sbs.boxenplot(x='Condition', y='Distance', data=distance_grouped, color='blue', width=0.4)
#plt.legend()
#plt.show()

In [ ]:
df_300_grouped = clean_complete_df.groupby(['Frame', 'Condition'])['Distance'].mean().reset_index()

In [ ]:
df_300_grouped = df_300_grouped.pivot(index='Frame', columns='Condition', values='Distance')

In [ ]:
ax = sbs.lineplot(data = clean_complete_df, x = 'Frame', y = 'Distance', hue = 'Condition')
plt.ylim(0,10)
plt.show()

In [ ]:
df_300 = clean_complete_df[clean_complete_df['Frame']<300]

In [ ]:
df_300_103 = df_300[(df_300["Condition"] == "10-2 cVA in Ethanol") | (df_300["Condition"] == "Ethanol 15L") | (df_300["Condition"] == "Water 15L")]

In [ ]:
ax = sbs.lineplot(data = df_300_103, x = 'Frame', y = 'Distance', hue = 'Condition')
plt.ylim(0,10)
plt.show()

In [ ]:
def retention_value(df, column_name, constant):
    """
    Compare a column in a pandas DataFrame to a constant and assign values based on the comparison.
    
    Parameters:
        df (pandas.DataFrame): The DataFrame containing the column to compare.
        column_name (str): The name of the column to compare.
        constant (float): The constant to compare against.
        
    Returns:
        pandas.DataFrame: The original DataFrame with a new column 'Comparison' containing the assigned values.
    """
    def compare_value(value):
        if value < constant:
            return 1
        else:
            return 0
    
    df['Retention Value'] = df[column_name].apply(compare_value)
    return df


In [ ]:
retention_value(clean_complete_df, 'Distance', 2.5)

In [ ]:
retention_df = clean_complete_df.groupby(['Condition', 'Frame'])['Retention Value'].mean()

In [ ]:
retention_df = retention_df.reset_index()

In [ ]:
ax = sbs.boxplot(data=retention_df, x = "Condition", y = "Retention Value")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')  # Rotate labels by 45 degrees

plt.show()

In [ ]:
sbs.lineplot(clean_complete_df, y = 'Retention Value', x = 'Frame', hue = 'Condition')

In [ ]:
def compare_to_constant(df, column_name, constant, width):
    """
    Compare a column in a pandas DataFrame to a constant and assign values based on the comparison.
    
    Parameters:
        df (pandas.DataFrame): The DataFrame containing the column to compare.
        column_name (str): The name of the column to compare.
        constant (float): The constant to compare against.
        
    Returns:
        pandas.DataFrame: The original DataFrame with a new column 'Comparison' containing the assigned values.
    """
    def compare_value(value):
        if value < constant - width:
            return 1
        elif value > constant + width:
            return -1
        else:
            return 0
    
    df['Side'] = df[column_name].apply(compare_value)
    return df


In [ ]:
def pref_index_calc(frame_sub_df):            
    sum_side = frame_sub_df['Side'].sum()
    total_side = frame_sub_df['Side'].count()
    if total_side > 0:
        ret_index = sum_side/total_side
    else:
        ret_index = 0
    return ret_index

In [ ]:
side_df = clean_complete_df.copy()
side_df = compare_to_constant(side_df, "Y", 14.6, 2)

In [ ]:
pref_df = pd.DataFrame(
    {"Condition" : [],
     "Trial" : [],
     "Preference Index" : [],
     "Frame" : []    
     })

for cond in conditions:
    for trial in trials_dict[f'{cond}']:
        sub_df = side_df[(side_df['Condition'] == cond) & (side_df['Trial'] == trial)]
        pref_list = []
        frame_list = sub_df["Frame"].unique().tolist()
        for frame in frame_list:
              frame_sub_df = sub_df[(sub_df['Frame'] == frame)]
              ret_frame = pref_index_calc(frame_sub_df)
              pref_list.append(ret_frame)
        temp_pref_df = pd.DataFrame(
             {    "Condition" : cond,
                  "Trial" : trial,
                  "Preference Index" : pref_list,
                  "Frame" : frame_list
             }
        )
        pref_df = pd.concat([pref_df, temp_pref_df])

In [ ]:
# Retention index - bin data into >3cm and <3cm 

# for cond in conditions:
#     for trial in trials_dict[f'{cond}']:
#         sub_df = clean_complete_df[(clean_complete_df['Condition'] == cond) & (clean_complete_df['Trial'] == trial)]
#         list of retention indices per trial per condition - make it a series and then collate them all into a big dataframe
#         index by condition and frame letsgooo - generates means and standard deviation across x trials in yth frame
#         plot mean and std deviation of retention index as lineplot
#         for frame in frame_list:
#               frame_sub_df = sub_df[(sub_df['Frame'] == frame)]
#               sum_side = frame_sub_df['Distance'].sum()
#               total_side = sum_side = frame_sub_df['Distance'].count()
#               ret_index =          

In [ ]:
# Grouped Analysis for Preference

pref_grouped_df_mean = pref_df.groupby(['Condition', 'Trial'])["Preference Index"].mean()
retained_grouped_df_mean = pref_grouped_df_mean.reset_index()

#retained_grouped_df_mean = retained_grouped_df_mean.pivot(index = "Trial", columns= "Condition", values="Retention")

In [ ]:
#retained_grouped_df_mean = retained_grouped_df_mean.reset_index()
retained_grouped_df_mean.head()

In [ ]:
retained_grouped_df_mean.drop(columns=["Trial"],inplace=True)

In [ ]:
ax = sbs.boxplot(data = retained_grouped_df_mean, x = "Condition", y = "Preference Index")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')  # Rotate labels by 45 degrees

plt.show()

In [ ]:
retained_byframe = pref_df.groupby(["Condition", "Frame"])["Preference Index"].mean()

In [ ]:
retained_byframe = retained_byframe.reset_index()
retained_byframe.head()

In [ ]:
ax = sbs.lineplot(data = pref_df, x = "Frame", y = "Preference Index", hue="Condition", errorbar="se")
plt.show()

In [ ]:
pref_df_103 = pref_df[(pref_df["Condition"] == "Group - 10-3 EtAc Fed") | (pref_df["Condition"] == "Single - 10-3 EtAc Fed")]
pref_df_103 = pref_df_103[pref_df_103["Frame"] < 300]

In [ ]:
ax = sbs.lineplot(data = pref_df_103, x = "Frame", y = "Preference Index", hue="Condition", errorbar="se")
plt.show()

In [ ]:
pref_df_104 = pref_df[(pref_df["Condition"] == "Group - 10-4 EtAc Fed") | (pref_df["Condition"] == "Single - 10-4 EtAc Fed")]
pref_df_104 = pref_df_104[pref_df_104["Frame"] < 300]

In [ ]:
ax = sbs.lineplot(data = pref_df_104, x = "Frame", y = "Preference Index", hue="Condition", errorbar="se")
plt.show()

In [ ]:
pref_df_105 = pref_df[(pref_df["Condition"] == "Group - 10-5 EtAc Fed") | (pref_df["Condition"] == "Single - 10-5 EtAc Fed")]
pref_df_105 = pref_df_105[pref_df_105["Frame"] < 300]

In [ ]:
ax = sbs.lineplot(data = pref_df_105, x = "Frame", y = "Preference Index", hue="Condition", errorbar="se")
plt.show()

In [ ]:
pref_df_WT =                  pref_df[(pref_df["Condition"] == "WT Fed") 
                                    | (pref_df["Condition"] == "WT 5h") 
                                    | (pref_df["Condition"] == "WT 24h")
                                    ]



In [ ]:
pref_df_WT_600 = pref_df_WT[pref_df_WT['Frame'] > 300]

In [ ]:
ax = sbs.lineplot(data = pref_df_WT_600, x = "Frame", y = "Preference Index", hue="Condition", errorbar="se")
plt.show()

In [ ]:
ax = sbs.lineplot(data = pref_df_WT, x = "Frame", y = "Preference Index", hue="Condition", errorbar="se")
plt.show()

In [ ]:
retained_df_trh =               pref_df[(pref_df["Condition"] == "Trh Fed") 
                                    | (pref_df["Condition"] == "Trh 5h") 
                                    ]



In [ ]:
retained_df_trh_600 = retained_df_trh[retained_df_trh['Frame'] < 900]
retained_df_trh_600 = retained_df_trh_600[retained_df_trh_600['Frame'] > 300]

In [ ]:
ax = sbs.lineplot(data = retained_df_trh_600, x = "Frame", y = "Preference Index", hue="Condition", errorbar="se")
plt.show()

In [ ]:
ax = sbs.lineplot(data = retained_df_trh, x = "Frame", y = "Preference Index", hue="Condition", errorbar="se")
plt.show()

In [ ]:
#Plotting Y Axis vs Time

ax = sbs.lineplot(data = clean_complete_df, x = "Frame", y = "Y", hue="Condition", errorbar="se")
plt.show()

In [ ]:
y_df_trh =               clean_complete_df[(clean_complete_df["Condition"] == "Trh Fed") 
                                    | (clean_complete_df["Condition"] == "Trh 5h") 
                                    ]



In [ ]:
#Plotting Y Axis vs Time

ax = sbs.lineplot(data = y_df_trh, x = "Frame", y = "Y", hue="Condition", errorbar="se")
plt.show()

In [ ]:
y_df_wt =               clean_complete_df[(clean_complete_df["Condition"] == "WT Fed") 
                                    | (clean_complete_df["Condition"] == "WT 5h")
                                    | (clean_complete_df["Condition"] == "WT 24h") 
                                    ]



In [ ]:
#Plotting Y Axis vs Time

ax = sbs.lineplot(data = y_df_wt, x = "Frame", y = "Y", hue="Condition", errorbar="se")
plt.show()

In [ ]:
sbs.violinplot(data = retained_df_trh[retained_df_trh["Frame"]<300], x = "Condition", y = "Preference Index")

In [ ]:
sbs.violinplot(data = pref_df_WT[pref_df_WT["Frame"]<300], x = "Condition", y = "Preference Index")